# Construyendo el Almacén de Datos de la Liga

**El Problema:**
Ustedes son los Arquitectos de Datos de una agencia internacional de *scouting*. Cada fin de semana, el sistema transaccional (OLTP) de los estadios genera un archivo plano masivo (`players_data-2025_2026.csv`) con todos los eventos registrados. 

Si el director de la agencia intenta calcular métricas históricas conectándose directamente a este archivo crudo, el sistema colapsa por redundancia y falta de indexación analítica.

**Su Misión:**
Construir una tubería **ETL (Extract, Transform, Load)** que tome este archivo transaccional y lo convierta en un **Esquema de Estrella (OLAP)** en memoria. Al final, deberán ejecutar un cubo analítico (*Roll-up*) para responder una pregunta de negocio estratégica.

In [3]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/players_data-2025_2026.csv"
df_raw = pd.read_csv(url)

print("--- FASE 1: EXTRACT (Extracción Transaccional) ---")
print("Datos crudos extraídos del sistema OLTP:")
display(df_raw.head(3))

--- FASE 1: EXTRACT (Extracción Transaccional) ---
Datos crudos extraídos del sistema OLTP:


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,...,3,0,0,20,51,5,37,17,27,0
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0,0,1,0,1,0,0
2,3,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,0,0,0,3,1,0,3,2,4,0


---
### FASE 2: TRANSFORM (Modelado del Esquema de Estrella)

El archivo plano tiene demasiado texto mezclado con métricas. Vamos a separarlo arquitectónicamente. Primero, crearemos nuestras **Tablas de Dimensión** (El Contexto).

**Reto 1:** Extraigan los datos únicos para crear una dimensión de Equipos y una dimensión de Jugadores. Generen IDs numéricos artificiales para optimizar la base analítica.

In [4]:
print("--- FASE 2A: TRANSFORM (Tablas de Dimensión) ---")

dim_equipo = df_raw[['Squad', 'Comp']].drop_duplicates().reset_index(drop=True)
dim_equipo = dim_equipo.rename(columns={'Squad': 'equipo', 'Comp': 'liga'})
dim_equipo['id_equipo'] = np.arange(1, len(dim_equipo) + 1)
dim_equipo = dim_equipo[['id_equipo', 'equipo', 'liga']]

dim_jugador = df_raw[['Player', 'Pos', 'Age']].drop_duplicates().reset_index(drop=True)
dim_jugador = dim_jugador.rename(columns={'Player': 'jugador', 'Pos': 'posicion', 'Age': 'edad'})
dim_jugador['id_jugador'] = np.arange(1, len(dim_jugador) + 1)
dim_jugador = dim_jugador[['id_jugador', 'jugador', 'posicion', 'edad']]

print("Dimensión Equipo (Muestra):")
display(dim_equipo.head(3))
print("\nDimensión Jugador (Muestra):")
display(dim_jugador.head(3))

--- FASE 2A: TRANSFORM (Tablas de Dimensión) ---
Dimensión Equipo (Muestra):


,id_equipo,equipo,liga
0,1,Leeds United,eng Premier League
1,2,Wolves,eng Premier League
2,3,Nottingham Forest,eng Premier League



Dimensión Jugador (Muestra):


,id_jugador,jugador,posicion,edad
0,1,Brenden Aaronson,"MF,FW",24.0
1,2,Jerome Abbey,MF,15.0
2,3,Zach Abbott,DF,19.0


Ahora construiremos el corazón del Data Warehouse: **La Tabla de Hechos (`Fact_Rendimiento`)**.
Recuerden la regla de oro: La tabla de hechos NO debe contener texto. Solo debe tener los IDs de las dimensiones que acabamos de crear y las métricas numéricas puras (goles, asistencias, minutos).

In [5]:
print("--- FASE 2B: TRANSFORM (Tabla de Hechos) ---")

df_temp = df_raw.merge(dim_equipo, left_on='Squad', right_on='equipo', how='left')
df_temp = df_temp.merge(dim_jugador, left_on='Player', right_on='jugador', how='left')

fact_rendimiento = df_temp[['id_equipo', 'id_jugador', 'Min', 'Gls', 'Ast']].copy()
fact_rendimiento = fact_rendimiento.rename(columns={
    'Min': 'minutos',
    'Gls': 'goles',
    'Ast': 'asistencias'
})

print("Tabla de Hechos (Métrica Pura):")
display(fact_rendimiento.head(3))

--- FASE 2B: TRANSFORM (Tabla de Hechos) ---
Tabla de Hechos (Métrica Pura):


,id_equipo,id_jugador,minutos,goles,asistencias
0,1,1,2449,4,5
1,2,2,17,0,0
2,3,3,164,0,0


---
### FASE 3: LOAD Y CONSULTA OLAP (Roll-up)

Nuestra arquitectura está lista. Hemos separado el almacenamiento para que sea eficiente. 

De pronto, el Director Técnico hace una pregunta que requiere un análisis multidimensional masivo (*Roll-up*):
*"¿Cuál es la Liga de fútbol que tiene la mayor cantidad de Goles Totales acumulados, y cuántos minutos en promedio juegan los jugadores de esa liga?"*

**Reto Final:** 
Para responder, no pueden usar el archivo `df_raw`. Deben simular el comportamiento de un motor OLAP: unan la Tabla de Hechos con la Dimensión Equipos y ejecuten una agrupación.

In [6]:
print("--- FASE 3: CUBO OLAP (Roll-up Analítico) ---")

cubo_olap = fact_rendimiento.merge(dim_equipo[['id_equipo', 'liga']], on='id_equipo', how='left')

analisis_ligas = (
    cubo_olap.groupby('liga', as_index=False)
    .agg(
        goles_totales=('goles', 'sum'),
        minutos_promedio=('minutos', 'mean')
    )
    .sort_values('goles_totales', ascending=False)
)

analisis_ligas['minutos_promedio'] = analisis_ligas['minutos_promedio'].round(2)

print("Respuesta Analítica para el Director Técnico:")
display(analisis_ligas)

--- FASE 3: CUBO OLAP (Roll-up Analítico) ---
Respuesta Analítica para el Director Técnico:


,liga,goles_totales,minutos_promedio
1,eng Premier League,1055,1335.66
2,es La Liga,1025,1204.61
0,de Bundesliga,985,1180.44
4,it Serie A,944,1203.69
3,fr Ligue 1,856,1049.90


---
### 🧠 Defensa del Diseño Arquitectónico (Responde aquí)

Como ingenieros de datos, acaban de transformar un archivo plano en un modelo de estrella y luego volvieron a unir las tablas para hacer la consulta OLAP. 

1. Si al final tuvimos que hacer un `merge()` (JOIN) para responder la pregunta del Director, ¿por qué nos tomamos la molestia de separar los datos en Hechos y Dimensiones en lugar de dejar el `df_raw` gigante en el servidor analítico desde el principio? 
2. ¿Qué ventaja física (de disco o memoria) tiene aislar las columnas de texto (`player_name`, `team_name`) en dimensiones separadas?

> **Tu Respuesta:** 
1. Hice el merge() al final porque necesitaba unir la información útil para responder la pregunta, pero sin trabajar con el archivo gigante completo. La separación entre hechos y dimensiones me ayuda a mantener los datos más ordenados y eficientes para analizar.
2. Guardar texto en dimensiones separadas me ahorra espacio, porque no repito nombres de equipos o jugadores en cada fila. Así la tabla de hechos queda más pequeña, más rápida y más fácil de consultar.